<a href="https://colab.research.google.com/github/Viggo-Kristensen/kaggle-competitions/blob/main/f1-pit-stop-prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Problem Definition**

### Predict whether a f1 car is going in for a pit stop on a given lap.

## **2. Imports**

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# Metrics
from sklearn.metrics import roc_auc_score



## **3. Dataset**

Each row represents a specific car on a given lap. The target variable `PitNextLap` indicates whether a car will pit on the following lap. Features include `Lap_Timing`, `RaceProgress`, `Tyre_state` and positional information.

### Uploading Files

In [2]:
!mkdir -p /root/.kaggle
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory


In [3]:
!pip install -q kaggle

# upload kaggle.json ONCE
from google.colab import files
files.upload()

!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c playground-series-s6e5

Saving kaggle.json to kaggle.json
playground-series-s6e5.zip: Skipping, found more recently modified local copy (use --force to force download)


### Unzipping Files

In [5]:
!unzip -q playground-series-s6e5.zip -d data

replace data/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


### Creating DataFrames

In [6]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [7]:
target = "PitNextLap"

drop_cols = ["id", "Driver", target]

X = train.drop(columns=drop_cols)
y = train[target]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "bool"]).columns

## **4. Feature Engineering**

In [8]:
class AddFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["Pit_Window_Signal"] = X["LapNumber"] / (X["TyreLife"] + 1e-6)
        X["Laps_Left"] = (1 - X["RaceProgress"]) / (X["RaceProgress"] / X["LapNumber"])
        X["Total_Laps"] = 1 / (X["RaceProgress"] / X["LapNumber"])

        return X

## **5. Preprocessing**

### Numeric Pipeline

In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### Categorical Pipeline

In [10]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

### Preprocessor

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    remainder="passthrough"
)

## **6. Pipeline**

In [12]:
model = Pipeline(steps=[
    ("feature_engineering", AddFeatures()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

## **7. Cross Validation**

In [13]:
scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc", n_jobs=-1)

print("fold scores:", scores)
print("mean roc-auc", scores.mean())

fold scores: [0.85812425 0.86096983 0.86048823 0.8590806  0.86045316]
mean roc-auc 0.8598232152801139


## **8. Training**

In [14]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppr

Pipeline(steps=[('feature_engineering', AddFeatures()),
                ('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Compound', 'Race'], dtype='object'))])),
                ('classifier', LogisticRegression(max_iter=1000))])

## **9. Feature Importance Analysis**

In [16]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = model.named_steps["classifier"].coef_[0]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

importance_df["Abs_Coefficient"] = importance_df["Coefficient"].abs()

importance_df = importance_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

print(importance_df.head())

                             Feature  Coefficient  Abs_Coefficient
2                     num__LapNumber     4.881604         4.881604
9                  num__RaceProgress    -4.595453         4.595453
15                 cat__Compound_WET    -1.656699         1.656699
32  cat__Race_Mexico City Grand Prix    -1.278317         1.278317
21      cat__Race_Belgian Grand Prix     1.110498         1.110498


## **10. Evaluation**

In [17]:
val_probs = model.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, val_probs)

np.float64(0.8615262111019975)

## **11. Submission**

In [18]:
submission = pd.read_csv("data/sample_submission.csv")
submission["PitNextLap"] = model.predict_proba(test)[:, 1]
submission.to_csv("submission.csv", index=False)

## **12. Reflections**

**Notes**


*   Dropped the Driver category since the weights were all near 0

*   Among the engineered features `Laps_Left` proved to be the strongest predictor.

*   The final model achieved a public leaderboard ROC-AUC score of 0.885, placing #840 on the Kaggle leaderboard at submission time.




**What i learned**


*   Building end to end pipelines using scikit-learn


*   Using cross validation to evaluate models more reliably


*   Evaluating the usability of specific features using coefficients

*   Understanding the effect of L2 regularisation

*   Designing high impact features for linear models








